In [ ]:
import time
import copy
import numpy as np
import joblib
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from pathlib import Path
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.utils.class_weight import compute_sample_weight
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE


In [ ]:
BASE_ENV = Path().resolve().parent
ART= BASE_ENV / 'artifacts'

In [ ]:
X_train = np.load(ART / 'datasets/train/X.npy')
y_train = np.load(ART / 'datasets/train/y.npy')
X_val = np.load(ART / 'datasets/val/X.npy')
y_val = np.load(ART / 'datasets/val/y.npy')
X_test = np.load(ART / 'datasets/test/X.npy')
y_test = np.load(ART / 'datasets/test/y.npy')


In [ ]:

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

In [ ]:
encoder = joblib.load(ART / 'preprocessors/encoder.joblib')
num_of_classes = len(encoder.classes_)

In [ ]:
print("Before SMOTE:")
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(encoder.inverse_transform(unique), counts):
    print(f"  {cls}: {cnt}")

# Define target counts for minority classes only
# Don't touch Benign and Generic — they're already huge
sampling_strategy = {}
target_counts = {
    'Rare_Attack':  8000,
    'DoS':      15000,   # was 5000
    'Shellcode': 5000,   # was 2000
    'Worms':     3000,   # was 1000
}

In [ ]:
for cls, target in target_counts.items():
    if cls in encoder.classes_:
        encoded_label = encoder.transform([cls])[0]
        current_count = np.sum(y_train == encoded_label)
        if current_count < target:
            sampling_strategy[encoded_label] = target


In [ ]:
sm = SMOTE(
    sampling_strategy=sampling_strategy,
    random_state=42,
    k_neighbors=3  # small because some classes have very few samples
)

In [ ]:
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

In [ ]:
print("\nAfter SMOTE:")
unique, counts = np.unique(y_train_res, return_counts=True)
for cls, cnt in zip(encoder.inverse_transform(unique), counts):
    print(f"  {cls}: {cnt}")

In [ ]:
X_proto, _, y_proto, _ = train_test_split(
    X_train_res, y_train_res, 
    train_size=0.05,           # Keep exactly 5%
    stratify=y_train_res,  # CRITICAL: Maintain the exact ratio of attack types
    random_state=42
)

In [ ]:
param_grid = {
    'max_depth': [3, 6, 9],              # Shallow vs Deep trees
    'learning_rate': [0.01, 0.1, 0.3],   # Slow vs Fast learning
    'n_estimators': [50, 100, 200],      # Number of trees
    'subsample': [0.8, 1.0]              # Prevent overfitting by using 80% of data per tree
}

In [ ]:
xgb_tuner = XGBClassifier(objective='multi:softprob', random_state=42, n_jobs=2)

# Set up the search engine
random_search = RandomizedSearchCV(
    estimator=xgb_tuner,
    param_distributions=param_grid,
    n_iter=10,             # Test exactly 10 random combinations from the menu
    scoring='accuracy',
    cv=3,                  # Cross-validation: double checks the score 3 times per test
    verbose=2,             # Prints progress so you aren't staring at a blank screen
    random_state=42,
    n_jobs=2                # Use all CPU cores
)

In [ ]:
random_search.fit(X_proto, y_proto)
print("Best Hyperparameters:", random_search.best_params_)

In [ ]:
xgb_final = XGBClassifier(
    objective='multi:softprob', 
    num_class=num_of_classes,
    max_depth=9,             # Injected from Tuner
    learning_rate=0.1,       # Injected from Tuner
    n_estimators=100,         # Injected from Tuner
    subsample=0.8,           # Injected from Tuner
    random_state=42,
    n_jobs=-1                # Use all CPU cores
)

In [ ]:
xgb_final.fit(X_train_res, y_train_res)

In [ ]:
y_pred = xgb_final.predict(X_test)

In [ ]:
y_test_labels = encoder.inverse_transform(y_test)
y_pred_labels = encoder.inverse_transform(y_pred)

In [ ]:
accuracy_score(y_test_labels, y_pred_labels)

In [ ]:
print("\nDetailed Classification Report:")
print(classification_report(y_test_labels, y_pred_labels, digits=4))

In [ ]:
param_grid = {
    'max_depth':        [3, 6, 9],
    'learning_rate':    [0.01, 0.1, 0.3],
    'n_estimators':     [50, 100, 200],
    'subsample':        [0.8, 1.0],
    'num_leaves':       [31, 63, 127],  # LightGBM specific — controls tree complexity
    'class_weight':     ['balanced', None]            # Prevent overfitting by using 80% of data per tree
}

In [ ]:
lgbm_tuner = LGBMClassifier(objective='multiclass', random_state=42, n_jobs=2)

random_search = RandomizedSearchCV(
    estimator=lgbm_tuner,
    param_distributions=param_grid,
    n_iter=10,
    scoring='accuracy',
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=2
)

In [ ]:
random_search.fit(X_proto, y_proto)
print("Best Hyperparameters:", random_search.best_params_)

In [ ]:
lgbm_final = LGBMClassifier(
    objective='multiclass',
    num_class=num_of_classes,
    subsample=0.8,
    num_leaves=127,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.01,
    random_state=42,
    n_jobs=-1
)

In [ ]:
lgbm_final.fit(X_train_res, y_train_res)

In [ ]:
y_pred = lgbm_final.predict(X_test)

In [ ]:
y_test_labels = encoder.inverse_transform(y_test)
y_pred_labels = encoder.inverse_transform(y_pred)

In [ ]:
accuracy_score(y_test_labels, y_pred_labels)

In [ ]:
print("\nDetailed Classification Report:")
print(classification_report(y_test_labels, y_pred_labels, digits=4))

In [ ]:
xgb_final.save_model(ART / 'models/xgb_NUSW-NB15.json')

In [ ]:
# The exact 41 features that survived the Hard Mode purge
hard_mode_features = hard_mode_features = ['dur',
                    'sbytes',
                    'dbytes',
                    'sttl',
                    'dttl',
                    'sloss',
                    'dloss',
                    'Sload',
                    'Dload',
                    'Spkts',
                    'Dpkts',
                    'swin',
                    'stcpb',
                    'smeansz',
                    'dmeansz',
                    'Sjit',
                    'Djit',
                    'Sintpkt',
                    'Dintpkt',
                    'tcprtt',
                    'synack',
                    'ackdat',
                    'is_sm_ips_ports',
                    'ct_state_ttl',
                    'ct_srv_dst',
                    'ct_dst_ltm',
                    'ct_src_ ltm',
                    'ct_dst_sport_ltm',
                    'proto_esp',
                    'proto_igmp',
                    'proto_udp',
                    'proto_udt',
                    'state_CON',
                    'state_ECR',
                    'state_FIN',
                    'state_INT',
                    'state_PAR',
                    'state_TST',
                    'state_URN',
                    'state_no',
                    'service_ftp-data']
# Safety check to ensure the array shape matches the name list
if X_train.shape[1] != len(hard_mode_features):
    print(f"WARNING: Feature count mismatch! Array has {X_train.shape[1]} columns, but list has {len(hard_mode_features)}.")
else:
    # Extract the importance scores from your Hard Mode model
    importances = xgb_final.feature_importances_

    # Map the scores using our hardcoded list instead of df.columns
    feature_importance_df = pd.DataFrame({
        'Feature': hard_mode_features,
        'Importance': importances
    })

    # Sort to find the new "Cheat Codes"
    feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

    print("Top 10 Features Driving the 100% Accuracy:")
    print(feature_importance_df.head(10))

    # Visualize it
    plt.figure(figsize=(10, 6))
    sns.barplot(
        x='Importance', 
        y='Feature', 
        data=feature_importance_df.head(10), 
        palette='viridis'
    )
    plt.title('XGBoost Hard Mode Feature Importances', fontweight='bold')
    plt.xlabel('Importance Score')
    plt.ylabel('Network Feature')
    plt.tight_layout()
    plt.show()

In [ ]:
# The exact 41 features that survived the Hard Mode purge
hard_mode_features = ['dur',
                    'sbytes',
                    'dbytes',
                    'sttl',
                    'dttl',
                    'sloss',
                    'dloss',
                    'Sload',
                    'Dload',
                    'Spkts',
                    'Dpkts',
                    'swin',
                    'stcpb',
                    'smeansz',
                    'dmeansz',
                    'Sjit',
                    'Djit',
                    'Sintpkt',
                    'Dintpkt',
                    'tcprtt',
                    'synack',
                    'ackdat',
                    'is_sm_ips_ports',
                    'ct_state_ttl',
                    'ct_srv_dst',
                    'ct_dst_ltm',
                    'ct_src_ ltm',
                    'ct_dst_sport_ltm',
                    'proto_esp',
                    'proto_igmp',
                    'proto_udp',
                    'proto_udt',
                    'state_CON',
                    'state_ECR',
                    'state_FIN',
                    'state_INT',
                    'state_PAR',
                    'state_TST',
                    'state_URN',
                    'state_no',
                    'service_ftp-data']

# Safety check to ensure the array shape matches the name list
if X_train.shape[1] != len(hard_mode_features):
    print(f"WARNING: Feature count mismatch! Array has {X_train.shape[1]} columns, but list has {len(hard_mode_features)}.")
else:
    # Extract the importance scores from your Hard Mode model
    importances = lgbm_final.feature_importances_

    # Map the scores using our hardcoded list instead of df.columns
    feature_importance_df = pd.DataFrame({
        'Feature': hard_mode_features,
        'Importance': importances
    })

    # Sort to find the new "Cheat Codes"
    feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

    print("Top 10 Features Driving the 100% Accuracy:")
    print(feature_importance_df.head(10))

    # Visualize it
    plt.figure(figsize=(10, 6))
    sns.barplot(
        x='Importance', 
        y='Feature', 
        data=feature_importance_df.head(10), 
        palette='viridis'
    )
    plt.title('XGBoost Hard Mode Feature Importances', fontweight='bold')
    plt.xlabel('Importance Score')
    plt.ylabel('Network Feature')
    plt.tight_layout()
    plt.show()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [ ]:
batch_size = 256
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

validation_dataset = TensorDataset(X_val_tensor, y_val_tensor)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class IDSNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(IDSNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)

        self.fc2 = nn.Linear(hidden_dim, hidden_dim//2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)  

        self.output_layer = nn.Linear(hidden_dim//2, output_dim)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.output_layer(x)
        return x

In [ ]:
input_dim = X_train.shape[1]
hidden_dim = 128
output_dim = num_of_classes
model = IDSNN(input_dim, hidden_dim, output_dim).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

In [ ]:
model.eval()

In [ ]:
all_predictions, all_true_labels = [], []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        # Move test batch to GPU
        batch_X = batch_X.to(device)
        
        # Get raw mathematical outputs (logits) from the network
        outputs = model(batch_X)
        
        # Find the neuron with the highest probability (the prediction)
        # torch.max returns both the value and the index. We only want the index [1]
        _, predicted_classes = torch.max(outputs, 1)
        
        # Move predictions back to the CPU memory to work with Scikit-Learn
        all_predictions.extend(predicted_classes.cpu().numpy())
        all_true_labels.extend(batch_y.numpy())

y_test_labels = encoder.inverse_transform(all_true_labels)
y_pred_labels = encoder.inverse_transform(all_predictions)

In [ ]:
accuracy_score(y_test_labels, y_pred_labels)

In [ ]:
print(classification_report(y_test_labels, y_pred_labels, digits=4))

In [ ]:
torch.save(model.state_dict(), ART / 'models/dnn_UNSW-NB15.pth')

In [ ]:
def create_sliding_windows(X, y, window_size=5):
    X_windows, y_windows = [], []
    for i in range(len(X) - window_size + 1):
        X_windows.append(X[i:i+window_size])
        y_windows.append(y[i+window_size-1])  # Label of the last item in the window
    return np.array(X_windows), np.array(y_windows)

In [ ]:
X_train_windows, y_train_windows = create_sliding_windows(X_train, y_train, window_size=10)
X_val_windows, y_val_windows = create_sliding_windows(X_val, y_val, window_size=10)

In [ ]:
X_train_windows.shape

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
X_train_tensor = torch.tensor(X_train_windows, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_windows, dtype=torch.long).to(device)

X_val_tensor = torch.tensor(X_val_windows, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val_windows, dtype=torch.long).to(device)

In [ ]:
batch_size = 256
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_dataset = TensorDataset(X_val_tensor, y_val_tensor)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class IDSNN_LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2):
        super(IDSNN_LSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2 if num_layers > 1 else 0.0
            )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        lstm_out = lstm_out[:, -1, :]  # Take the output of the last time step
        lstm_out = self.fc(lstm_out)
        return lstm_out

In [ ]:
input_dim = X_train_windows.shape[2]  
hidden_dim = 128
output_dim = num_of_classes
model = IDSNN_LSTM(input_dim, hidden_dim, output_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
EPOCHS = 20
PATIENCE = 5

best_val_loss = float('inf')
patience_counter = 0
best_model_state = copy.deepcopy(model.state_dict())

In [ ]:
start_time = time.time()

for epoch in range(EPOCHS):
    epoch_time = time.time()

    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in validation_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item()
    avg_val_loss = val_loss / len(validation_loader)

    epoch_duration = time.time() - epoch_time
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Time: {epoch_duration:.2f}s")
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, ART / 'models/best_idsnn_lstm.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered!")
            break
total_training_time = (time.time() - start_time) / 60
print(f"Total Training Time: {total_training_time:.2f} minutes")

In [ ]:
X_test_windows, y_test_windows = create_sliding_windows(X_test, y_test, window_size=10) 
X_test_tensor = torch.tensor(X_test_windows, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_windows, dtype=torch.long)

In [ ]:
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
model.load_state_dict(torch.load(ART / 'models/best_idsnn_lstm.pth'))
model.to(device)

In [ ]:
model.eval()

In [ ]:
lstm_predictions, lstm_true_labels = [], []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        _, predicted_classes = torch.max(outputs, 1)
        lstm_predictions.extend(predicted_classes.cpu().numpy())
        lstm_true_labels.extend(batch_y.numpy())
lstm_true_labels = encoder.inverse_transform(lstm_true_labels)
lstm_pred_labels = encoder.inverse_transform(lstm_predictions)

In [ ]:
accuracy_score(lstm_true_labels, lstm_pred_labels)

In [ ]:
print(classification_report(lstm_true_labels, lstm_pred_labels, digits=4))

In [ ]:
old_file = ART / 'models/best_idsnn_lstm.pth'
new_file = ART / 'models/lstm_UNSW-NB15.pth'

if old_file.exists():
    old_file.rename(new_file)
    print(f"Successfully renamed to: {new_file.name}")
else:
    print("The old file does not exist. It may have already been renamed.")